<a href="https://colab.research.google.com/github/sdesai38/Projects/blob/main/ECGR6116MiniProject1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
Problem statment: single machine scheduling w/ weighted tardiness

A single machine  must process a set of n = 10 tasks J = {1,2,...,10}
each task i belongs to the set J is charachterized by:
processing time pi > 0,
deadline di belongs to set alll positive real numbers
taridness wieght wi > 0

task are processed start to finish, must run to completion once started, one task at at time limit
if task i completes at time Ci, its weighted tardiness penalty is defined as li(Ci) = wimax(0,Ci-d-)

goal determine an ordering schedule of the 10 tasks that minimizes tht total weighted tardiness
'''

'''
Prompt utilized:
Given the following, write a script to use Djikstras algorithm on the following:
 given a set of tasks J ={1:10} i belongs to J goal: sum (i = 1:10) Li(Ci)
 where Ci is defined as Ci = ti + pi. where ti is the sum of all previous
 processing time and pi is the processing time for task i define nodes as a
 state V = (S, ti), define source node cost as 0
 define edges: (S,t) --> (S U J, t+pi) define edge
  weights : Li(Ci) = wi(max(0, ti + pi - di)) where
   wi is the tardiness weight and di is the deadline
'''

'\nPrompt utilized:\nGiven the following, write a script to use Djikstras algorithm on the following:\n given a set of tasks J ={1:10} i belongs to J goal: sum (i = 1:10) Li(Ci) \n where Ci is defined as Ci = ti + pi. where ti is the sum of all previous \n processing time and pi is the processing time for task i define nodes as a \n state V = (S, ti), define source node cost as 0 \n define edges: (S,t) --> (S U J, t+pi) define edge\n  weights : Li(Ci) = wi(max(0, ti + pi - di)) where\n   wi is the tardiness weight and di is the deadline\n'

In [ ]:
import heapq

def dijkstra_weighted_tardiness(processing_times, deadlines, weights):
    """
    Solve: min sum w_i * max(0, C_i - d_i)
    using Dijkstra on state space.

    processing_times: dict {job: p_i}
    deadlines: dict {job: d_i}
    weights: dict {job: w_i}
    """

    jobs = list(processing_times.keys())
    n = len(jobs)

    # Priority queue: (cost, current_time, completed_set)
    pq = []
    heapq.heappush(pq, (0, 0, frozenset(),()))

    # Store best known cost for (S, t)
    visited = {}

    while pq:
        cost, t, S, sequence = heapq.heappop(pq)

        # Goal condition: all jobs completed
        if len(S) == n:
            return cost, sequence

        # Avoid revisiting worse states
        if (S) in visited and visited[(S)] <= cost:
            continue

        visited[(S)] = cost

        # Expand to all unscheduled jobs
        for j in jobs:
            if j not in S:
                new_S = frozenset(S | {j})
                new_t = t + processing_times[j]

                # Completion time Cj = new_t
                tardiness = max(0, new_t - deadlines[j])
                edge_cost = weights[j] * tardiness

                new_cost = cost + edge_cost\

                new_sequence = sequence + (j,)

                heapq.heappush(pq, (new_cost, new_t, new_S, new_sequence))

    return None  # No solution

#modified to add sequence tracker to track optimal order of tasks
#reduced storage of t in V = (S,t) as in single machine the time is determined by S, helps speed up calculation


In [ ]:
def edf_with_weighted_tardiness(processing_times, deadlines, weights):
    """
    Earliest Deadline First schedule
    + compute total weighted tardiness

    processing_times: dict {job: p_i}
    deadlines: dict {job: d_i}
    weights: dict {job: w_i}

    Returns:
        ordered_jobs
        schedule_info (job, start, completion, tardiness, weighted_tardiness)
        total_weighted_tardiness
    """


    ordered_jobs = sorted(deadlines.keys(), key=lambda j: deadlines[j])

    schedule_info = []
    current_time = 0
    total_cost = 0


    for job in ordered_jobs:
        start_time = current_time
        completion_time = current_time + processing_times[job]

        tardiness = max(0, completion_time - deadlines[job])
        weighted_tardiness = weights[job] * tardiness

        total_cost += weighted_tardiness

        schedule_info.append(
            (job, start_time, completion_time, tardiness, weighted_tardiness)
        )

        current_time = completion_time

    return ordered_jobs, schedule_info, total_cost



In [ ]:
import itertools

def compute_weighted_tardiness(order, processing_times, deadlines, weights):
    """
    Compute total weighted tardiness for a given job order.
    """
    current_time = 0
    total_cost = 0

    for job in order:
        completion_time = current_time + processing_times[job]
        tardiness = max(0, completion_time - deadlines[job])
        total_cost += weights[job] * tardiness
        current_time = completion_time

    return total_cost


def brute_force_optimal_schedule(processing_times, deadlines, weights):
    """
    Try all permutations to find the optimal schedule.

    Returns:
        best_order
        best_cost
    """

    jobs = list(processing_times.keys())
    best_cost = float("inf")
    best_order = None

    for perm in itertools.permutations(jobs):
        cost = compute_weighted_tardiness(
            perm, processing_times, deadlines, weights
        )

        if cost < best_cost:
            best_cost = cost
            best_order = perm

    return best_order, best_cost

In [ ]:
# ================================
# Djikstra's algorithm
# ================================
if __name__ == "__main__":

    # J = {1,...,10}
    processing_times = {
        1: 2,
        2: 4,
        3: 3,
        4: 5,
        5: 2,
        6: 6,
        7: 3,
        8: 4,
        9: 1,
       10: 2
    }

    deadlines = {
        1: 6,
        2: 9,
        3: 7,
        4: 12,
        5: 5,
        6: 14,
        7: 10,
        8: 11,
        9: 4,
       10: 8
    }

    weights = {
        1: 3,
        2: 2,
        3: 6,
        4: 4,
        5: 8,
        6: 2,
        7: 5,
        8: 3,
        9: 9,
       10: 4
    }

    optimal_cost, sequence = dijkstra_weighted_tardiness(
        processing_times,
        deadlines,
        weights
    )


print("Optimal total weighted tardiness:", optimal_cost)
print("Optimal task ordering:", sequence)

Optimal total weighted tardiness: 153
Optimal task ordering: (9, 5, 3, 10, 7, 1, 4, 8, 2, 6)


In [ ]:
#Earliest-deadline-first
order, schedule, total_cost = edf_with_weighted_tardiness(
        processing_times,
        deadlines,
        weights
    )

print("EDF Order:", order)
print("\nDetailed Schedule:")
for job, start, completion, T, wT in schedule:
        print(f"Job {job}: "
              f"Start={start}, "
              f"Completion={completion}, "
              f"Tardiness={T}, "
              f"Weighted={wT}")
print("\nTotal Weighted Tardiness:", total_cost)

EDF Order: [9, 5, 1, 3, 10, 2, 7, 8, 4, 6]

Detailed Schedule:
Job 9: Start=0, Completion=1, Tardiness=0, Weighted=0
Job 5: Start=1, Completion=3, Tardiness=0, Weighted=0
Job 1: Start=3, Completion=5, Tardiness=0, Weighted=0
Job 3: Start=5, Completion=8, Tardiness=1, Weighted=6
Job 10: Start=8, Completion=10, Tardiness=2, Weighted=8
Job 2: Start=10, Completion=14, Tardiness=5, Weighted=10
Job 7: Start=14, Completion=17, Tardiness=7, Weighted=35
Job 8: Start=17, Completion=21, Tardiness=10, Weighted=30
Job 4: Start=21, Completion=26, Tardiness=14, Weighted=56
Job 6: Start=26, Completion=32, Tardiness=18, Weighted=36

Total Weighted Tardiness: 181


In [ ]:
best_order, best_cost = brute_force_optimal_schedule(
        processing_times,
        deadlines,
        weights
    )

print("Optimal Order:", best_order)
print("Optimal Total Weighted Tardiness:", best_cost)

Optimal Order: (5, 9, 3, 10, 7, 1, 4, 8, 2, 6)
Optimal Total Weighted Tardiness: 153
